# Cognitive & Behavioral Predictors of Student Outcomes

**Group Members:** Mohammad Zoraiz (mz248), Rithik Duvva (rrd18), Kartikeye Gupta (kg330), Ratish Korrapati (rsk49), Enoch Li (yel6)  
**Course:** CS216 Everything Data (Spring 2026)  
**Date:** Feb 19, 2026  
**Repo:** [CampusSignals](https://github.com/zoraizmohammad/CampusSignals)  

## AI Disclosure
Artificial Intelligence Tools: ChatGPT (OpenAI, ChatGPT Edu via Duke University, accessed Jan–Feb 2026)  
- Conceptualization: refined and narrowed research questions.  
- Information collection: located documented, non-synthetic datasets and documentation.  
- Methodology support: suggested mediation-style analysis + supervised ML with ablation testing.  
- Writing/review/editing: improved structure and clarity.

---

## Research Questions
1. How do screen-use patterns (including type), sleep characteristics, and behavioral indicators relate to academic outcomes (grades/GPA proxies)?
2. To what extent do sleep characteristics mediate the relationship between screen use and academic outcomes?
3. Can we predict student outcomes from behavioral and sleep-related features, and which feature groups contribute most?

---

## Modules (mapped to this notebook)
- **Visualization (Module 3):** distributions, relationships, stratified plots, model diagnostics.
- **Data Wrangling (Module 4):** missingness handling, encoding, feature engineering.
- **Statistical Inference (Module 7):** multivariable regression + mediation-style inference with uncertainty.
- **Prediction & Supervised ML (Module 8):** cross-validation, ablations, and interpretability/feature importance.

---

## Notebook Output
This notebook runs the same analysis workflow independently on:
- **Dataset A:** `datasets/mendeley/repositorio.xlsx` (adolescents; screen use, sleep, daytime sleepiness, grades)
- **Dataset B:** `datasets/student-life/` (college students; passive sensing + sleep/stress + academic performance)

Then it compares results across cohorts (directionality + predictive performance).

---

## Plan of Attack (sections)
- Dataset A: ingestion -> schema validation -> feature engineering -> EDA -> mediation -> prediction -> ablations
- Dataset B: ingestion -> cohort outcome assembly -> feature engineering (including keyword-based app categories) -> EDA -> mediation -> prediction -> ablations
- Cross-cohort comparison, robustness, and final reproducibility cleanup


## Dataset A (Adolescents)

### A.1 Ingestion & Schema Validation
- Read `datasets/mendeley/repositorio.xlsx`
- Identify columns for screen-use (weekday/weekend, before bedtime), sleep metrics, daytime sleepiness/PDSS, and grades/GPA

### A.2 Wrangling & Feature Engineering
- Clean missingness
- Engineer screen-use features + sleep features
- Create outcome target: averaged GPA proxy = mean(`DI1 lengua`, `DJ1 matematica`)

### A.3 EDA & Visualization
- Distributions, relationships, stratified comparisons

### A.4 Inference / Mediation-Style Workflow
- screen -> sleep -> academic outcome

### A.5 Prediction & Ablations (RQ3)
- Cross-validation prediction with feature-group ablations

---

## Dataset B (College Students)

### B.1 Ingestion (cohort outcome + sleepiness inputs)
- Load `datasets/student-life/dataset/education/grades.csv`
- Load sleepiness survey inputs (e.g., `datasets/student-life/dataset/survey/psqi.csv`)
- Load EMA “Sleep” JSONs (response folder)

### B.2 Feature Engineering
- Passive sensing aggregation into user-level features
- Keyword-based app-to-screen-category mapping (keyword rules defined in-notebook)

### B.3 EDA & Visualization
- Sanity checks and exploratory relationships

### B.4 Inference / Mediation-Style Workflow
- screen -> sleep -> academic outcome

### B.5 Prediction & Ablations (RQ3)
- Cross-validation prediction with feature-group ablations

---

## Cross-Cohort Comparison
- Compare effect directions and predictive performance between Dataset A and B
- Robustness checks + final limitations


In [ ]:
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd


def _colref_to_col_index(cell_ref: str) -> int:
    """Convert Excel cell ref like 'A1' or 'BC12' to 0-based column index."""
    letters = re.match(r"^[A-Z]+", cell_ref.upper()).group(0)
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord('A') + 1)
    return idx - 1


def read_xlsx_first_sheet(xlsx_path: str | Path) -> pd.DataFrame:
    """Read the first sheet from an .xlsx using only ZIP/XML parsing.

    This avoids needing `openpyxl`/`xlrd` in the runtime.
    """
    xlsx_path = Path(xlsx_path)
    with zipfile.ZipFile(xlsx_path, 'r') as zf:
        # Shared strings are used for most cell values.
        shared_strings = {}
        if 'xl/sharedStrings.xml' in zf.namelist():
            shared_xml = zf.read('xl/sharedStrings.xml')
            shared_root = ET.fromstring(shared_xml)
            ns_ss = {'w': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for idx, si in enumerate(shared_root.findall('.//w:si', ns_ss)):
                parts = []
                for t in si.findall('.//w:t', ns_ss):
                    if t.text:
                        parts.append(t.text)
                shared_strings[idx] = ''.join(parts)

        # Find sheet target (sheet xml file) for the first sheet.
        workbook_xml = zf.read('xl/workbook.xml')
        workbook_root = ET.fromstring(workbook_xml)
        ns_wb = {'w': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        sheets = []
        for sheet in workbook_root.findall('.//w:sheets/w:sheet', ns_wb):
            name = sheet.attrib.get('name')
            rid = sheet.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
            sheets.append((name, rid))
        if not sheets:
            raise ValueError('No sheets found in workbook.xml')

        first_name, first_rid = sheets[0]

        rels_xml = zf.read('xl/_rels/workbook.xml.rels')
        rels_root = ET.fromstring(rels_xml)
        ns_rel = {'r': 'http://schemas.openxmlformats.org/package/2006/relationships'}
        relmap = {}
        for rel in rels_root.findall('.//r:Relationship', ns_rel):
            rid = rel.attrib.get('Id')
            target = rel.attrib.get('Target')
            relmap[rid] = target

        if first_rid not in relmap:
            raise KeyError(f'Could not resolve relationship id {first_rid}')

        target = relmap[first_rid]
        # Target should look like 'worksheets/sheet1.xml'
        if target.startswith('worksheets/'):
            sheet_xml_path = 'xl/' + target
        else:
            # Sometimes it's already fully qualified under xl/
            sheet_xml_path = target
            if not sheet_xml_path.startswith('xl/'):
                sheet_xml_path = 'xl/' + sheet_xml_path

        sheet_xml_bytes = zf.read(sheet_xml_path)
        sheet_root = ET.fromstring(sheet_xml_bytes)
        ns_sheet = {'w': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        rows = sheet_root.findall('.//w:sheetData/w:row', ns_sheet)
        if not rows:
            raise ValueError('No rows found in sheetData')

        # Header row assumed to be first row.
        header_row = rows[0]
        header_cells = header_row.findall('w:c', ns_sheet)
        if not header_cells:
            raise ValueError('No header cells in first row')

        col_indices = []
        for c in header_cells:
            ref = c.attrib.get('r')
            if not ref:
                continue
            col_indices.append(_colref_to_col_index(ref))
        max_col = max(col_indices) + 1

        headers = [''] * max_col
        for c in header_cells:
            cell_ref = c.attrib.get('r', '')
            col_i = _colref_to_col_index(cell_ref)
            t = c.attrib.get('t')
            v = c.find('w:v', ns_sheet)
            if v is None or v.text is None:
                val = ''
            else:
                if t == 's':
                    val = shared_strings.get(int(v.text), '')
                else:
                    val = v.text
            headers[col_i] = (val or '').strip()

        def cell_value(c):
            t = c.attrib.get('t')
            v = c.find('w:v', ns_sheet)
            if v is None or v.text is None:
                return ''
            if t == 's':
                return shared_strings.get(int(v.text), '')
            return v.text

        data = []
        for row in rows[1:]:
            row_vals = [''] * max_col
            for c in row.findall('w:c', ns_sheet):
                ref = c.attrib.get('r')
                if not ref:
                    continue
                col_i = _colref_to_col_index(ref)
                row_vals[col_i] = cell_value(c)
            # Keep rows even if entirely empty; we’ll drop later.
            data.append(row_vals)

        df = pd.DataFrame(data, columns=headers)

        # Drop fully-empty columns/rows.
        df = df.replace(r'^\s*$', np.nan, regex=True)
        df = df.dropna(axis=1, how='all')
        df = df.dropna(axis=0, how='all')

        # Strip column names
        df.columns = [str(c).strip() for c in df.columns]

        print(f"Loaded sheet '{first_name}' with shape={df.shape}")
        return df


# -----------------------
# Dataset A ingestion
# -----------------------

a_path = Path('datasets/mendeley/repositorio.xlsx')
assert a_path.exists(), f"Missing {a_path}"

# Load raw spreadsheet (prefer cached CSV if present for speed)
cache_csv = Path('datasets/mendeley/repositorio.csv')
if cache_csv.exists():
    print(f"Loading cached Dataset A CSV from {cache_csv}")
    A_raw = pd.read_csv(cache_csv)
else:
    print("Loading Dataset A from XLSX (ZIP/XML parsing)")
    A_raw = read_xlsx_first_sheet(a_path)

# Quick schema validation + variable-group mapping
cols = set(map(str, A_raw.columns))

# Outcomes (per plan: lingua/matematica -> averaged GPA proxy)
outcome_cols = [c for c in ['lengua', 'matematica'] if c in cols]

# Daytime sleepiness / PDSS proxy
pdss_cols = [c for c in cols if c.lower().strip() == 'somno01pdss' or 'pdss' in c.lower()]

# Screen-use related columns (heuristics based on Spanish question/variable names)
screen_cols = sorted(
    [c for c in cols if re.search(r'pantalla|videojueg|redes|tvonline|tpotv|otras|tpopantallatotal', c, flags=re.I)]
)

# Sleep-related columns
sleep_cols = sorted(
    [c for c in cols if re.search(r'dorm|siesta|sueñ|acost|levant|tard|horasueñ|hora.*dorm|bedtime', c, flags=re.I)]
)

print("\nSchema validation")
print("- outcome_cols found:", outcome_cols)
print("- pdss_cols found:", pdss_cols)

missing = []
if len(outcome_cols) != 2:
    missing.append(f"outcomes missing: {set(['lengua','matematica']) - set(outcome_cols)}")
if len(pdss_cols) < 1:
    missing.append('somno01pdss / pdss column missing')

if missing:
    print("WARNING:")
    for m in missing:
        print("-", m)
else:
    print("All required outcome + PDSS columns are present.")

print("\nVariable-group mapping (sample of matched columns)")

def _print_group(title, columns, limit=25):
    preview = columns[:limit]
    print(f"- {title}: {preview}{' ...' if len(columns) > limit else ''} (n={len(columns)})")

# Screen categories (gaming/social/TV/other) via name prefixes.
cat_gaming = sorted([c for c in screen_cols if re.search(r'videojueg|juego|tpojueg', c, flags=re.I)])
cat_social = sorted([c for c in screen_cols if re.search(r'redes', c, flags=re.I)])
cat_tv = sorted([c for c in screen_cols if re.search(r'tvonline|tpotv|TV o Smart TV|smart', c, flags=re.I)])
cat_other = sorted([c for c in screen_cols if c not in set(cat_gaming + cat_social + cat_tv)])

_print_group('screen_cols (broad)', screen_cols)
_print_group('screen category - gaming', cat_gaming)
_print_group('screen category - social', cat_social)
_print_group('screen category - tv/video', cat_tv)
_print_group('screen category - other', cat_other)
_print_group('sleep_cols (heuristic)', sleep_cols)

print("\nA_raw.head()")
A_raw.head()
